# Full Comparison — All 4 Pipelines

Loads **all trained models** and produces a comprehensive side-by-side comparison across all four approaches.

| Group | Models | Approach | Data |
|-------|--------|----------|------|
| Old Baseline | EfficientNet-B0 V4 | No seg, old model | cropped_classification_data |
| Hard Crop (Seg) | EffB3, ResNet-50, ConvNeXt | Seg → crop wound → classify | new_cropped_data |
| No Seg | EffB3, ResNet-50, ConvNeXt | Raw image → classify | TRAINTEST |
| Overlay (Seg+Context) | EffB3, ResNet-50, ConvNeXt | Seg → dim background → classify | overlay_data |

**Overlay vs Hard Crop:** Both use segmentation. Hard crop throws away context (only wound region fed to classifier). Overlay keeps the full image but dims background 50% — wound is highlighted, nothing lost.

**Prerequisites:** Notebooks 06_5, 07, 08, 10 must be fully run.

## Cell 1: Imports & Paths

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, datasets, models
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_curve, auc, confusion_matrix)
import seaborn as sns

BASE        = r'C:\Users\PC\Desktop\GraduationProject\MyProjectSTILL\FirstTry'
MODEL_DIR   = os.path.join(BASE, 'outputs', 'models')
VIZ_DIR     = os.path.join(BASE, 'outputs', 'visualizations')
RAW_CLS     = os.path.join(BASE, 'TRAINTEST')
NEW_CROPPED = os.path.join(BASE, 'outputs', 'new_cropped_data')
OLD_CROPPED = os.path.join(BASE, 'outputs', 'cropped_classification_data')
OVERLAY_DIR = os.path.join(BASE, 'outputs', 'overlay_data')
os.makedirs(VIZ_DIR, exist_ok=True)

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE   = 224
BATCH_SIZE = 32

plt.rcParams.update({
    'figure.facecolor':'#0d1117', 'axes.facecolor':'#161b22',
    'axes.edgecolor':'#30363d',   'axes.labelcolor':'#c9d1d9',
    'text.color':'#c9d1d9',       'xtick.color':'#8b949e',
    'ytick.color':'#8b949e',      'grid.color':'#21262d', 'font.size':11
})
ACCENT, GREEN, BLUE, YELLOW, PURPLE, ORANGE = '#e94560','#3fb950','#58a6ff','#e3b341','#bc8cff','#ff9f43'

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

print(f'Device : {DEVICE}')
print('Imports OK')

## Cell 2: Model Architecture Definitions

In [ ]:
# ---- Old EfficientNet-B0 V4 ----
class WoundClassifierV4(nn.Module):
    def __init__(self, num_classes=2, dropout_rate=0.45):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=None)
        nf = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate), nn.Linear(nf,256), nn.BatchNorm1d(256), nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate), nn.Linear(256,64),  nn.BatchNorm1d(64),  nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate*0.5), nn.Linear(64, num_classes)
        )
    def forward(self, x): return self.classifier(self.backbone(x))

# ---- Shared classifier architecture (used by notebooks 07, 08, 10) ----
def build_efficientnet_b3(nc=2):
    m=models.efficientnet_b3(weights=None); nf=m.classifier[1].in_features
    m.classifier=nn.Sequential(nn.Dropout(0.4),nn.Linear(nf,512),nn.BatchNorm1d(512),nn.ReLU(True),
                                nn.Dropout(0.3),nn.Linear(512,128),nn.BatchNorm1d(128),nn.ReLU(True),
                                nn.Dropout(0.2),nn.Linear(128,nc)); return m

def build_resnet50(nc=2):
    m=models.resnet50(weights=None); nf=m.fc.in_features
    m.fc=nn.Sequential(nn.Dropout(0.4),nn.Linear(nf,512),nn.BatchNorm1d(512),nn.ReLU(True),
                       nn.Dropout(0.3),nn.Linear(512,128),nn.BatchNorm1d(128),nn.ReLU(True),
                       nn.Dropout(0.2),nn.Linear(128,nc)); return m

def build_convnext_tiny(nc=2):
    m=models.convnext_tiny(weights=None); nf=m.classifier[2].in_features
    m.classifier[2]=nn.Sequential(nn.Dropout(0.4),nn.Linear(nf,512),nn.BatchNorm1d(512),nn.ReLU(True),
                                   nn.Dropout(0.3),nn.Linear(512,128),nn.BatchNorm1d(128),nn.ReLU(True),
                                   nn.Dropout(0.2),nn.Linear(128,nc)); return m

print('Architecture definitions ready.')

## Cell 3: Load All Models

In [ ]:
def load_cls(builder, path):
    m = builder().to(DEVICE)
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    sd = ck['model_state_dict'] if 'model_state_dict' in ck else ck
    m.load_state_dict(sd)
    m.eval()
    return m

loaded  = {}
missing = []

model_specs = [
    # (group,            display_name,        builder,              filename)
    ('Old Baseline', 'EffB0-V4 (old)',        WoundClassifierV4,    'classification_efficientnet_v4.pth'),
    ('Hard Crop',    'EffB3+HardCrop',        build_efficientnet_b3,'cls_withseg_efficientnetb3.pth'),
    ('Hard Crop',    'ResNet50+HardCrop',     build_resnet50,        'cls_withseg_resnet50.pth'),
    ('Hard Crop',    'ConvNeXt+HardCrop',     build_convnext_tiny,   'cls_withseg_convnext.pth'),
    ('No Seg',       'EffB3-NoSeg',           build_efficientnet_b3,'cls_noseg_efficientnetb3.pth'),
    ('No Seg',       'ResNet50-NoSeg',        build_resnet50,        'cls_noseg_resnet50.pth'),
    ('No Seg',       'ConvNeXt-NoSeg',        build_convnext_tiny,   'cls_noseg_convnext.pth'),
    ('Overlay',      'EffB3+Overlay',         build_efficientnet_b3,'cls_overlay_efficientnetb3.pth'),
    ('Overlay',      'ResNet50+Overlay',      build_resnet50,        'cls_overlay_resnet50.pth'),
    ('Overlay',      'ConvNeXt+Overlay',      build_convnext_tiny,   'cls_overlay_convnext.pth'),
]

for group, disp, builder, fname in model_specs:
    path = os.path.join(MODEL_DIR, fname)
    if os.path.exists(path):
        loaded[(group, disp)] = load_cls(builder, path)
        print(f'  Loaded  : {disp}')
    else:
        missing.append(disp)
        print(f'  MISSING : {disp}  ({fname})')

print(f'\n{len(loaded)}/10 models loaded.')
if missing:
    print(f'Missing: {missing}')
    print('Run the prerequisite notebooks first (06_5, 07, 08, 10).')

## Cell 4: Evaluate All Models

In [ ]:
def make_loader(root):
    ds = datasets.ImageFolder(root, transform=val_tf)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    return ds, dl

loaders = {
    'Old Baseline': make_loader(os.path.join(OLD_CROPPED, 'test')),
    'Hard Crop'   : make_loader(os.path.join(NEW_CROPPED, 'test')),
    'No Seg'      : make_loader(os.path.join(RAW_CLS, 'test')),
    'Overlay'     : make_loader(os.path.join(OVERLAY_DIR, 'test')),
}

print('Test sets:')
for g, (ds, dl) in loaders.items():
    print(f'  {g:<16} {len(ds)} images  classes={ds.classes}')


def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            out  = model(imgs)
            prob = torch.softmax(out, 1).cpu().numpy()
            all_preds.extend(prob.argmax(1))
            all_labels.extend(labels.numpy())
            all_probs.extend(prob[:, 0])
    lbl  = np.array(all_labels)
    pred = np.array(all_preds)
    prob = np.array(all_probs)
    fpr, tpr, _ = roc_curve(lbl, prob, pos_label=0)
    return {
        'acc' : accuracy_score(lbl, pred),
        'f1'  : f1_score(lbl, pred, average='macro', zero_division=0),
        'prec': precision_score(lbl, pred, average='macro', zero_division=0),
        'rec' : recall_score(lbl, pred, average='macro', zero_division=0),
        'auc' : auc(fpr, tpr),
        'fpr' : fpr, 'tpr': tpr,
        'lbl' : lbl, 'pred': pred,
        'cm'  : confusion_matrix(lbl, pred),
    }

results = {}
print('\nEvaluating all models...')
for (group, disp), model in loaded.items():
    _, dl = loaders[group]
    r = evaluate(model, dl)
    results[(group, disp)] = r
    print(f'  {disp:<24}  Acc={r["acc"]*100:.2f}%  F1={r["f1"]:.4f}  AUC={r["auc"]:.4f}')

print('\nEvaluation complete.')

## Cell 5: Full Comparison Table

In [ ]:
def grade(v):
    if v >= 0.90: return 'Excellent'
    if v >= 0.80: return 'Good'
    if v >= 0.65: return 'Fair'
    return 'Needs work'

sorted_results = sorted(results.items(), key=lambda x: x[1]['f1'], reverse=True)

print(f'\n{"="*80}')
print(f'  FULL COMPARISON — ALL 10 MODELS (sorted by F1)')
print(f'{"="*80}')
print(f'  {"Group":<16} {"Model":<26} {"Acc%":>7} {"F1":>7} {"Prec":>7} {"Rec":>7} {"AUC":>7}')
print(f'  {"-"*73}')

prev_group = None
for (group, disp), r in sorted_results:
    if group != prev_group:
        print()
        prev_group = group
    print(f'  {group:<16} {disp:<26} {r["acc"]*100:>7.2f} {r["f1"]:>7.4f} '
          f'{r["prec"]:>7.4f} {r["rec"]:>7.4f} {r["auc"]:>7.4f}')

print(f'\n{"="*80}')
best_key = max(results, key=lambda k: results[k]['f1'])
print(f'  OVERALL BEST: {best_key[1]}  ({best_key[0]})  F1={results[best_key]["f1"]:.4f}')

# Group averages
print(f'\n{"="*55}')
print(f'  {"Group":<18} {"Avg F1":>8} {"Avg Acc%":>10} {"Models":>8}')
print(f'  {"-"*46}')
for group in ['Old Baseline', 'Hard Crop', 'No Seg', 'Overlay']:
    vals = [(r['f1'], r['acc']) for (g,d),r in results.items() if g==group]
    if vals:
        avg_f1  = np.mean([v[0] for v in vals])
        avg_acc = np.mean([v[1] for v in vals])
        print(f'  {group:<18} {avg_f1:>8.4f} {avg_acc*100:>10.2f} {len(vals):>8}')
print(f'{"="*55}')

## Cell 6: Side-by-Side Bar Chart — With vs Without Segmentation

In [ ]:
arch_names  = ['EfficientNet-B3', 'ResNet-50', 'ConvNeXt-Tiny']
crop_keys   = [('Hard Crop','EffB3+HardCrop'), ('Hard Crop','ResNet50+HardCrop'), ('Hard Crop','ConvNeXt+HardCrop')]
noseg_keys  = [('No Seg','EffB3-NoSeg'),       ('No Seg','ResNet50-NoSeg'),       ('No Seg','ConvNeXt-NoSeg')]
overlay_keys= [('Overlay','EffB3+Overlay'),    ('Overlay','ResNet50+Overlay'),    ('Overlay','ConvNeXt+Overlay')]

metric_keys   = ['acc', 'f1', 'auc']
metric_labels = ['Accuracy', 'F1 Score', 'AUC-ROC']

fig, axes = plt.subplots(1, 3, figsize=(20, 7), facecolor='#0d1117')
fig.suptitle('Hard Crop vs No-Seg vs Overlay — 3 Architectures', color='#c9d1d9', fontsize=15, y=1.02)

x = np.arange(len(arch_names))
w = 0.22

for ax, mk, ml in zip(axes, metric_keys, metric_labels):
    ax.set_facecolor('#161b22'); ax.spines[:].set_edgecolor('#30363d')

    crop_vals    = [results[k][mk] for k in crop_keys    if k in results]
    noseg_vals   = [results[k][mk] for k in noseg_keys   if k in results]
    overlay_vals = [results[k][mk] for k in overlay_keys if k in results]

    b1 = ax.bar(x - w,   crop_vals,    w, label='Hard Crop',  color=YELLOW,  alpha=0.88)
    b2 = ax.bar(x,       noseg_vals,   w, label='No Seg',     color=GREEN,   alpha=0.88)
    b3 = ax.bar(x + w,   overlay_vals, w, label='Overlay',    color=BLUE,    alpha=0.88)

    for bar, v in list(zip(b1,crop_vals)) + list(zip(b2,noseg_vals)) + list(zip(b3,overlay_vals)):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.004,
                f'{v:.3f}', ha='center', va='bottom', fontsize=7.5, color='#c9d1d9')

    ax.set_xticks(x)
    ax.set_xticklabels(arch_names, rotation=10, ha='right')
    ax.set_ylim(0, 1.18)
    ax.set_title(ml, color='#c9d1d9', fontsize=13)
    ax.legend(facecolor='#21262d', labelcolor='#c9d1d9', fontsize=9)
    ax.axhline(0.80, color='#444d56', linewidth=0.8, linestyle='--')

plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR,'comparison_all_pipelines.png'), dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved -> outputs/visualizations/comparison_all_pipelines.png')

## Cell 7: ROC Curves — All Groups

In [ ]:
group_colors = {
    'Old Baseline': [PURPLE],
    'Hard Crop'   : [YELLOW, '#ffd32a', '#ffa502'],
    'No Seg'      : [GREEN,  '#26de81', '#20bf6b'],
    'Overlay'     : [BLUE,   '#45aaf2', '#2d98da'],
}

fig, axes = plt.subplots(1, 4, figsize=(24, 6), facecolor='#0d1117')
fig.suptitle('ROC Curves — All 4 Groups', color='#c9d1d9', fontsize=14)

for ax, group in zip(axes, ['Old Baseline', 'Hard Crop', 'No Seg', 'Overlay']):
    ax.set_facecolor('#161b22'); ax.spines[:].set_edgecolor('#30363d')
    ax.plot([0,1],[0,1], color='#444d56', linewidth=1, linestyle='--', label='Random')

    group_items = [(disp, r) for (g, disp), r in results.items() if g == group]
    cols = group_colors[group]
    for i, (disp, r) in enumerate(group_items):
        ax.plot(r['fpr'], r['tpr'], color=cols[i % len(cols)], linewidth=2,
                label=f"{disp.split('+')[-1] if '+' in disp else disp}  {r['auc']:.3f}")

    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(group, color='#c9d1d9', fontsize=12)
    ax.legend(facecolor='#21262d', labelcolor='#c9d1d9', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR,'comparison_roc_all.png'), dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved -> outputs/visualizations/comparison_roc_all.png')

## Cell 8: Full Dashboard — Final Scorecard

In [ ]:
gp_colors_map = {
    'Old Baseline': PURPLE,
    'Hard Crop'   : YELLOW,
    'No Seg'      : GREEN,
    'Overlay'     : BLUE,
}

fig = plt.figure(figsize=(24, 14), facecolor='#0d1117')
fig.suptitle('Complete Comparison Dashboard — All 10 Models', color='#c9d1d9', fontsize=16, fontweight='bold', y=0.99)
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

bar_cols  = [gp_colors_map.get(g, '#888888') for (g, d), _ in sorted_results]
all_names = [d for (g, d), _ in sorted_results]
all_accs  = [r['acc'] for (_, r) in sorted_results]
all_f1s   = [r['f1']  for (_, r) in sorted_results]
all_aucs  = [r['auc'] for (_, r) in sorted_results]

# 1. Accuracy horizontal bar
ax1 = fig.add_subplot(gs[0, 0]); ax1.set_facecolor('#161b22')
bars = ax1.barh(all_names, all_accs, color=bar_cols, alpha=0.88)
for bar, v in zip(bars, all_accs):
    ax1.text(v+0.005, bar.get_y()+bar.get_height()/2, f'{v*100:.1f}%', va='center', fontsize=7.5, color='#c9d1d9')
ax1.set_xlim(0, 1.18); ax1.axvline(0.80, color='#444d56', linewidth=0.8, linestyle='--')
ax1.set_title('Accuracy', color='#c9d1d9'); ax1.spines[:].set_edgecolor('#30363d')
patches = [Patch(color=gp_colors_map[g], label=g) for g in ['Old Baseline','Hard Crop','No Seg','Overlay']]
ax1.legend(handles=patches, facecolor='#21262d', labelcolor='#c9d1d9', fontsize=8)

# 2. F1 horizontal bar
ax2 = fig.add_subplot(gs[0, 1]); ax2.set_facecolor('#161b22')
bars2 = ax2.barh(all_names, all_f1s, color=bar_cols, alpha=0.88)
for bar, v in zip(bars2, all_f1s):
    ax2.text(v+0.005, bar.get_y()+bar.get_height()/2, f'{v:.4f}', va='center', fontsize=7.5, color='#c9d1d9')
ax2.set_xlim(0, 1.18); ax2.axvline(0.80, color='#444d56', linewidth=0.8, linestyle='--')
ax2.set_title('F1 Score', color='#c9d1d9'); ax2.spines[:].set_edgecolor('#30363d')

# 3. AUC horizontal bar
ax3 = fig.add_subplot(gs[0, 2]); ax3.set_facecolor('#161b22')
bars3 = ax3.barh(all_names, all_aucs, color=bar_cols, alpha=0.88)
for bar, v in zip(bars3, all_aucs):
    ax3.text(v+0.005, bar.get_y()+bar.get_height()/2, f'{v:.4f}', va='center', fontsize=7.5, color='#c9d1d9')
ax3.set_xlim(0, 1.18); ax3.axvline(0.80, color='#444d56', linewidth=0.8, linestyle='--')
ax3.set_title('AUC-ROC', color='#c9d1d9'); ax3.spines[:].set_edgecolor('#30363d')

# 4. Group avg F1 comparison
ax4 = fig.add_subplot(gs[1, 0]); ax4.set_facecolor('#161b22')
groups_ord = ['Old Baseline', 'Hard Crop', 'No Seg', 'Overlay']
grp_avg_f1 = []
for grp in groups_ord:
    vals = [r['f1'] for (g,d),r in results.items() if g==grp]
    grp_avg_f1.append(np.mean(vals) if vals else 0.0)   # 0.0 instead of nan
grp_cols = [gp_colors_map[g] for g in groups_ord]

x_pos = np.arange(len(groups_ord))
bars4 = ax4.bar(x_pos, grp_avg_f1, color=grp_cols, alpha=0.88)
for bar, v in zip(bars4, grp_avg_f1):
    ax4.text(bar.get_x()+bar.get_width()/2, v+0.005, f'{v:.4f}',
             ha='center', fontsize=10, color='#c9d1d9', fontweight='bold')
ax4.set_ylim(0, 1.1)
ax4.axhline(0.80, color='#444d56', linewidth=0.8, linestyle='--')
ax4.set_title('Average F1 per Group', color='#c9d1d9')
ax4.spines[:].set_edgecolor('#30363d')
# Fix: set_xticks first so set_xticklabels pairs with FixedLocator
ax4.set_xticks(x_pos)
ax4.set_xticklabels(groups_ord, rotation=10, ha='right')

# 5. Best model confusion matrix
best_key_sorted = sorted_results[0][0]
br = sorted_results[0][1]
ax5 = fig.add_subplot(gs[1, 1]); ax5.set_facecolor('#161b22')
cmap_cm = LinearSegmentedColormap.from_list('cm', ['#161b22','#0d419d','#58a6ff'])
sns.heatmap(br['cm'], annot=True, fmt='d', cmap=cmap_cm,
            xticklabels=['infected','non-inf'], yticklabels=['infected','non-inf'], ax=ax5,
            linewidths=0.5, linecolor='#21262d', annot_kws={'size':14,'weight':'bold','color':'#e6edf3'})
ax5.set_title(f'Best Model CM\n{best_key_sorted[1]}', color='#c9d1d9')
ax5.set_xlabel('Predicted', color='#8b949e'); ax5.set_ylabel('Actual', color='#8b949e')

# 6. Scorecard table
ax6 = fig.add_subplot(gs[1, 2]); ax6.set_facecolor('#161b22'); ax6.axis('off')
ax6.set_title('Final Scorecard', color='#c9d1d9', fontsize=13)
rows = [[d, g[:7], f'{r["acc"]*100:.1f}%', f'{r["f1"]:.4f}', grade(r["f1"])]
        for (g, d), r in sorted_results]
tbl = ax6.table(cellText=rows, colLabels=['Model','Group','Acc%','F1','Grade'],
                cellLoc='center', loc='center', bbox=[0,0,1,1])
tbl.auto_set_font_size(False); tbl.set_fontsize(7.5)
for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor('#21262d')
    if row == 0:
        cell.set_facecolor('#21262d'); cell.set_text_props(color=BLUE, fontweight='bold')
    else:
        cell.set_facecolor('#1c2128' if row%2==0 else '#161b22')
        cell.set_text_props(color='#c9d1d9')

plt.savefig(os.path.join(VIZ_DIR,'comparison_full_dashboard.png'), dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved -> outputs/visualizations/comparison_full_dashboard.png')

## Cell 9: Conclusion

In [ ]:
print(f'\n{"="*70}')
print('  FINAL CONCLUSIONS — ALL 4 PIPELINES')
print(f'{"="*70}')

bk = sorted_results[0][0]
br = sorted_results[0][1]
print(f'\n  Overall best model : {bk[1]}  ({bk[0]})')
print(f'  Accuracy  : {br["acc"]*100:.2f}%  ({grade(br["acc"])})')
print(f'  F1 Score  : {br["f1"]:.4f}  ({grade(br["f1"])})')
print(f'  AUC-ROC   : {br["auc"]:.4f}  ({grade(br["auc"])})')

# Group averages
groups_ord = ['Old Baseline', 'Hard Crop', 'No Seg', 'Overlay']
group_avgs = {}
for grp in groups_ord:
    f1s = [r['f1'] for (g,d),r in results.items() if g==grp]
    group_avgs[grp] = np.mean(f1s) if f1s else 0.0

print(f'\n  {"Pipeline":<20} {"Avg F1":>8}  Note')
print(f'  {"-"*58}')
notes = {
    'Old Baseline': 'Original system (EfficientNet-B0)',
    'Hard Crop'   : 'Seg → crop wound only → loses context',
    'No Seg'      : 'Raw image → classify directly',
    'Overlay'     : 'Seg → dim background → keeps context',
}
for grp in groups_ord:
    marker = ' ◄ BEST GROUP' if grp == max(group_avgs, key=group_avgs.get) else ''
    print(f'  {grp:<20} {group_avgs[grp]:>8.4f}  {notes[grp]}{marker}')

# Key finding: overlay vs noseg
ov_avg = group_avgs['Overlay']
ns_avg = group_avgs['No Seg']
hc_avg = group_avgs['Hard Crop']
delta_ov_ns = ov_avg - ns_avg
delta_ov_hc = ov_avg - hc_avg

print(f'\n  Key Findings:')
print(f'    Overlay vs No-Seg    : {delta_ov_ns:+.4f} F1  '
      f'({"Overlay WINS" if delta_ov_ns > 0 else "No-Seg wins"})')
print(f'    Overlay vs Hard Crop : {delta_ov_hc:+.4f} F1  '
      f'({"Overlay WINS" if delta_ov_hc > 0 else "Hard Crop wins"})')
print(f'    Hard Crop vs No-Seg  : {hc_avg-ns_avg:+.4f} F1  '
      f'(context loss {"confirmed harmful" if hc_avg < ns_avg else "was not an issue"})')

old_f1 = [r['f1'] for (g,d),r in results.items() if g=='Old Baseline'][0]
new_best_f1 = sorted_results[0][1]['f1']
print(f'\n  Improvement over old system: {new_best_f1-old_f1:+.4f} F1 '
      f'({(new_best_f1-old_f1)/old_f1*100:+.1f}%)')

print(f'\n{"="*70}')
print('  Generated files:')
for fname in ['comparison_all_pipelines.png','comparison_roc_all.png','comparison_full_dashboard.png']:
    print(f'    outputs/visualizations/{fname}')
print(f'{"="*70}')